In [24]:
import os
import sys 
os.chdir("/workspaces/dev/app")
sys.path.append("/workspaces/dev/app")

In [25]:
from services.whisper import WordService, WordParams, WordReturn
from services.whisper.Params import Hyperparameters
import librosa
import numpy as np
from silero_vad import load_silero_vad, get_speech_timestamps
from dotenv import load_dotenv
from pyannote.audio import Inference
from pyannote.core import Segment

In [26]:
load_dotenv()

True

In [27]:
HF_TOKEN=os.getenv("HF_TOKEN")

In [28]:
MODEL_SIZE = "large-v3"

SAMPLE_RATE = 16000
BUFFER_SIZE = 10

In [29]:
embedding_model = Inference( 
  model="pyannote/embedding",
  window="whole", 
  use_auth_token=HF_TOKEN,
  device="cuda"
)

/usr/local/lib/python3.10/dist-packages/pytorch_lightning/utilities/migration/migration.py:208: You have multiple `ModelCheckpoint` callback states in this checkpoint, but we found state keys that would end up colliding with each other after an upgrade, which means we can't differentiate which of your checkpoint callbacks needs which states. At least one of your `ModelCheckpoint` callbacks will not be able to reload the state.
[Kss]: Lightning automatically upgraded your loaded checkpoint from v1.2.7 to v2.5.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../root/.cache/torch/pyannote/models--pyannote--embedding/snapshots/4db4899737a38b2d618bbd74350915aa10293cb2/pytorch_model.bin`


Model was trained with pyannote.audio 0.0.1, yours is 3.3.2. Bad things might happen unless you revert pyannote.audio to 0.x.
Model was trained with torch 1.8.1+cu102, yours is 2.6.0+cu124. Bad things might happen unless you revert torch to 1.x.


/usr/local/lib/python3.10/dist-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['loss_func.W']


In [30]:
full_audio, sr = librosa.load("/workspaces/dev/.data/boda.wav", sr=SAMPLE_RATE)

In [31]:
# audio = audio[85 * SAMPLE_RATE:]

In [32]:
total_samples = len(full_audio)

segments = []
pos = 0
while pos < total_samples:
  rand_len = int(np.random.normal(loc=16000, scale=400))
  rand_len = np.clip(rand_len, 14000, 18000)
  end = min(pos + rand_len, total_samples)

  chunk = full_audio[pos:end]
  segments.append(chunk)
  pos = end

In [33]:
# full_text = ""
# for segment in segments:
#   if len(segment) < 160:
#     continue
#   seg, info = whisper.translate(segment, language="ko")
#   for s in seg:
#     full_text += s.text

In [34]:
# print(full_text)

In [35]:
HYPERPARAMETERS = {
  "weighted_prob_boundary": 0,
  "filter_by_duration_z": {
    "default": 2.0,
    "ko": 2.0,
    "en": 2.0,
  },
  "filter_by_probability": {
    "z": {
      "default": 2.0,
      "ko": 2.0,
      "en": 2.0,
    },
    "min_prob": {
      "default": 1.0,
      "ko": 0.4,
      "en": 0.4,
    },
  },
  "token_iou_padding": 0.2,
  "combine": {
    "search_range_time": {
      "default": 1.5,
      "ko": 1.5,
      "en": 1.5,
    },
    "threshold": {
      "default": 0.5,
      "ko": 0.25,
      "en": 0.5,
    },
    "tolerance": {
      "default": 0.3,
      "ko": 0.3,
      "en": 0.3,
    },
  },
  "refine_tolerance": {
    "default": 0.5,
    "ko": 0.5,
    "en": 0.5,
  }
}

In [36]:
MAX_PREV_TIME = 5

In [37]:
class TestWordService(WordService):
  def _get_weighted_probability(self, probabilities, start, end, duration, boundary):
    if boundary == 0:
      return probabilities
    center = (start + end) / 2
    if center > boundary:
      if center < duration - boundary:
        return probabilities
      return probabilities - probabilities * ((boundary - duration + center)/boundary) ** 2
    return probabilities - probabilities * (boundary - center/boundary) ** 2

In [38]:
hyper = Hyperparameters(None, HYPERPARAMETERS)

In [39]:
whisper_service = TestWordService.get_instance(MAX_PREV_TIME, hyper)

In [40]:
model = load_silero_vad(onnx=True)

In [41]:
def vad_audio(audio):
  timestamps = get_speech_timestamps(
    audio,
    model,
    sampling_rate=SAMPLE_RATE,
    threshold=0.4,
    min_silence_duration_ms = 400,
    speech_pad_ms=300
  )
  if timestamps is None:
    return None

  merged_audio = []

  for segment in timestamps:
    start = segment['start']
    end = segment['end']
    merged_audio.append(audio[start:end])

  return np.concatenate(merged_audio), timestamps

In [42]:
raise Exception("stop")

Exception: stop

In [ ]:
def recover_timeoffset(sentence, timestamps, original_duration):
  padding = 0
  tots=()
  for ts in timestamps:
    
  

In [ ]:
from IPython.display import Audio

In [ ]:
segment_id = 0
completed = {}
word_params = WordParams()

In [ ]:
segment = segments[segment_id]
segment_id += 1

va, ts = vad_audio(segment)
if va is not None:
  word_params.audio = va

  (result, audio) = whisper_service.transcribe(word_params)
  completed.update(result.completed_dict)

  print(f"{segment_id}" + "--" * 20)
  print([(v.lang, v.text) for k, v in completed.items()])
  print([(v.lang, v.text) for v in result.prev_words if v.is_word])
  print([(v.lang, v.text) for v in result.prev_recog if v.is_word])

  word_params.order = result.order
  word_params.time_offset = result.time_offset
  word_params.prev_audio = result.prev_audio
  word_params.prev_words = result.prev_words
  word_params.prev_recog = result.prev_recog
  word_params.prev_prob_mean = result.prev_prob_mean
  word_params.prev_prob_std = result.prev_prob_std
  word_params.prev_prob_count = result.prev_prob_count
  word_params.prev_dura_mean = result.prev_dura_mean
  word_params.prev_dura_std = result.prev_dura_std
  word_params.prev_dura_count = result.prev_dura_count

  if result.completed_dict:
    print("문장 및 구간 음성" + "-" * 20)
    for k, v in result.completed_dict.items():
      start = v.words[0].start
      end = v.words[-1].end
      print(f"{k} : {v.text}")
      print(f"\t {start} ~ {end}")

117----------------------------------------
[(['ko'], '오늘 뉴스를 본 것 같은데 네 우리나라 망원경을 발사 성공했죠'), (['ko'], '오늘? 네.'), (['ko'], '아까 낮에 발사가 다행히 성공을 했어요.'), (['ko'], '축하드립니다.'), (['ko'], '축하드립니다'), (['ko'], '미국 나사가 주도하는 거긴 한데 다른 나라 중에 우리나라가 유일하게 껴서 최종 목표가 앞으로 한 1, 2년 정도 안에 4억에서 5억 개 정도 은하를 지도로 보겠다는 겁니다.'), (['ko'], '불과 100년 전 정도까지만 해도 우리가 살고 있는 우리 은하가 그냥 우주의 전부다라고 생각을 한 거예요.'), (['ko'], '아마 단언컨대 우리나라 신문들을 포함해서 전 세계 모든 신문의 일면 사진이었을 거예요.'), (['ko'], '5, 6천 개 가까운 외계 행성을 찾았거든요.'), (['ko'], '외계 행성이 정말 흔하다는 걸 또 보여준 사례 중에 하나고 흥미로운 과학이야기 더욱 재밌게 전해드립니다.'), (['ko'], '반갑습니다.'), (['ko'], '과학을 보다 정용진입니다.'), (['ko'], '세종대학교에서 은하를 연구하고 학생을 가르치는 우주먼지 지웅배입니다.'), (['ko'], '성중간대학교 물리학과에서 학생을 우주먼지 지웅배입니다.'), (['ko'], '가르치고 연구는 요즘은 잘 안 하는 김범준입니다.'), (['ko'], '안녕하세요.'), (['ko'], '연세대학교 시스템 생물학과에서 미생물을 연구하고 있는 김흥민입니다.'), (['ko'], '네 안녕하세요.'), (['ko'], '경희대학교 약대에서 학생들 가르치고 연구하고 있는 백인하입니다.'), (['ko'], '오늘도 4분의 과학자와 함께 재밌는 과학 이야기 전해드립니다.'), (['ko'], '자 오늘도 그러면 재밌는 과학이야기 특히 오늘 현대 우주 탄생 100주년 얘기가 있어서 오늘 우주 얘기 좀 더 집중적으로 한

In [ ]:
index = 0
rt = list(result.completed_dict.items())
k, v = rt[index]
start = v.words[0].start * SAMPLE_RATE + 8000
end = v.words[-1].end * SAMPLE_RATE + 8000
print(f"{k} : {v.text}")
print(f"\t {start} ~ {end}")
Audio(full_audio[int(start):int(end)], rate=SAMPLE_RATE)

29 : 그래도 이제 다양한 질환에 쓰이는 게 대표적인 게 요즘 만나요.
	 1685732.0000000002 ~ 1753662.0000000002


In [ ]:
mr_ujmj = [(148061, 339589), (338760, 436145), (433746, 517335)]
mr_jyj = [(1013274, 1068548), (1071733, 1209101), (1209858, 1344749)]
mr_kbj = []
mr_kub = [(880324, 938541), ]
ms_bih = [(944284, 1009464)]

In [ ]:
Audio(result.prev_audio, rate=SAMPLE_RATE)

In [ ]:
for segment in segments:
  word_params.audio = segment

  (result, audio) = whisper_service.transcribe(word_params)
  completed.update(result.completed_dict)

  print(f"{segment_id}" + "--" * 20)
  print([(v.lang, v.text) for k, v in completed.items()])
  print([(v.lang, v.text) for v in result.prev_words if v.is_word])
  print([(v.lang, v.text) for v in result.prev_recog if v.is_word])

  word_params.order = result.order
  word_params.time_offset = result.time_offset
  word_params.prev_audio = result.prev_audio
  word_params.prev_words = result.prev_words
  word_params.prev_recog = result.prev_recog
  word_params.prev_prob_mean = result.prev_prob_mean
  word_params.prev_prob_std = result.prev_prob_std
  word_params.prev_prob_count = result.prev_prob_count
  word_params.prev_dura_mean = result.prev_dura_mean
  word_params.prev_dura_std = result.prev_dura_std
  word_params.prev_dura_count = result.prev_dura_count

0----------------------------------------
[]
[]
[('ko', ' BBC'), ('ko', ' 생방송'), ('ko', ' 인터뷰도')]
0----------------------------------------
[]
[]
[('ko', ' BBC'), ('ko', ' 생방송'), ('ko', ' 인터뷰도'), ('ko', ' 도중에'), ('ko', ' 자녀'), ('ko', ' 난입사건.')]
0----------------------------------------
[]
[]
[('ko', ' BBC'), ('ko', ' 생방송'), ('ko', ' 인터뷰도'), ('ko', ' 도중에'), ('ko', ' 자녀'), ('ko', ' 난입'), ('ko', ' 사건으로'), ('ko', ' 스타가'), ('ko', ' 된'), ('ko', ' 미국인'), ('ko', ' 교수가')]
0----------------------------------------
[]
[('ko', ' BBC'), ('ko', ' 생방송'), ('ko', ' 인터뷰도'), ('ko', ' 도중에')]
[('ko', ' 자녀'), ('ko', ' 난입'), ('ko', ' 사건으로'), ('ko', ' 스타가'), ('ko', ' 된'), ('ko', ' 미국인'), ('ko', ' 교수'), ('ko', ' 가족이'), ('ko', ' 오늘'), ('ko', ' 카메라'), ('ko', ' 앞에'), ('ko', ' 섰습니다.')]
0----------------------------------------
[]
[('ko', ' BBC'), ('ko', ' 생방송'), ('ko', ' 인터뷰도'), ('ko', ' 도중에'), ('ko', ' 자녀'), ('ko', ' 난입'), ('ko', ' 사건으로'), ('ko', ' 스타가')]
[('ko', ' 된'), ('ko', ' 미국인'), ('ko', ' 교수'), ('ko', ' 가족이

In [ ]:
for key, item in completed.items():
  print(key, item)
for v in result.prev_words:
  if v.is_word: print(v.text)
# for v in prev_recog:
#   print(v.text)

0 ['ko'] BBC 생방송 인터뷰도 도중에 자녀 난입 사건으로 스타가 된 미국인 교수 가족이 오늘 카메라 앞에 섰습니다.
1 ['ko'] 유튜브 스타가 된 4살짜리 딸은 이번엔 사탕을 입에 물고 배영진 기자입니다.
2 ['ko'] BBC 인터뷰 도중 딸과 아들의 등장으로 일약 스타가 된 로버트 켈리, 부산대 교수.
3 ['ko'] 유튜브 영상 감사합니다.
4 ['ko'] 건이 넘는 켈리 교수 가족은 세계적인 유명인사가 됐습니다.
5 ['ko'] 이렇게 언론의 관심이 커지자 켈리 교수 가족이 기자회견에 나섰습니다.
6 ['ko'] 춤을 췄던 첫째 딸 메리아는 사탕을 물었고 둘째 아들 존은 엄마 품에 안긴 모습이었습니다. 
7 ['en'] thought it was a disaster. 
8 ['en'] I immediately called texted or or emailed the BBC. 
9 ['en'] I communicated with the BBC immediately afterwards, and I apologized to them. 
10 ['en'] I said that if they never us back or never asked me to be on television again.
11 ['en'] I would understand.
12 ['ko'] 귀여운 춤으로 화제가 된 딸에 대한 질문도 잇따랐습니다.
13 ['en'] answered, that's She's four.
14 ['en'] She has no idea.
15 ['ko'] 당시 BBC와 인터뷰한 내용은 한국의 대통령 탄핵 사건이었습니다. 
16 ['en', 'ko'] months months millions of people on the streets, no one's car got burned The protesters even picked up their trash I find that that's just a model of a 일부 외국 네티즌들이 켈리 교수의